# Bloque 1 — Demo: Técnicas de Big Data

Este notebook acompaña la presentación del Bloque 1 y es el **notebook de referencia profundo**
para las técnicas de análisis sin IA de todo el curso: análisis de red (grafos, centralidad,
comunidades), TF-IDF y series temporales. Los casos de bloque 5 (CardingForums, HackingForums,
IronMarch) van a referenciar este notebook en vez de re-explicar cada técnica desde cero —
aquí es donde se ve el "cómo funciona" con calma; allá se aplica a cada foro concreto.

Usamos el dataset de **Carding Forums** como ejemplo porque es pequeño y limpio — perfecto para demos.

> **Prerrequisito**: haber ejecutado `bloque5_cardingforums/01_ingenieria_datos.ipynb` para tener los archivos `.parquet` limpios.

## Imports

Cargamos las librerías que vamos a usar.

- **pandas**: para manipular tablas de datos (leer archivos, filtrar filas, agrupar valores)
- **networkx**: para construir y analizar grafos (redes de usuarios)
- **plotly**: para visualizaciones interactivas
- **igraph** + **leidenalg**: para detectar comunidades con el algoritmo de Leiden (leidenalg trabaja sobre grafos de `igraph`, no de NetworkX directamente — por eso convertimos el grafo antes de usarlo)
- **sklearn.feature_extraction.text.TfidfVectorizer**: para calcular TF-IDF sobre el texto de los posts

In [1]:
import pandas as pd
import networkx as nx
import igraph as ig
import leidenalg
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

DATA_DIR = Path('../results')
print('Directorio de datos:', DATA_DIR.resolve())

Directorio de datos: /home/miguel/csbc26/results


## 1. Cargar los datos limpios

Cargamos los archivos `.parquet` generados en `bloque5_cardingforums/01_ingenieria_datos.ipynb`.

Un archivo **Parquet** es un formato de almacenamiento columnar muy eficiente — mucho más rápido de cargar que un CSV y ocupa menos espacio en disco. Pandas lo lee directamente con `pd.read_parquet()`.

In [ ]:
posts = pd.read_parquet(DATA_DIR / '01_posts_clean.parquet')
users = pd.read_parquet(DATA_DIR / '01_users_clean.parquet')

print(f'Posts cargados:   {len(posts):,}')
print(f'Usuarios cargados: {len(users):,}')
posts.head(3)

print('\nForos/leaks distintos en este dataset:', posts['forum'].nunique())
print(posts['forum'].value_counts())

### Por qué nos quedamos con un único foro (`Carder.pro_2013.04`)

Mirando la salida de arriba: este "dataset de CardingForums" en realidad **mezcla 5 leaks/foros
distintos** (columna `forum`), no es un dump único y limpio de una sola comunidad — son 5 volcados
independientes, con tamaños, ventanas temporales e idiomas muy distintos entre sí:

- `Carder.pro_2013.04`: 533.201 posts, 14.930 usuarios, 2009-10-29 → 2013-04-05, 96% texto válido.
- `Crdshop.su_2016.11.07`: 498.427 posts, 12.810 usuarios, hasta noviembre de 2016.
- `Carders.cc_2010.12`: 373.328 posts, 4.730 usuarios, 2009-02 → 2010-12.
- `Cardersplanet.biz_2010.05`: solo 4.075 posts / 788 usuarios.
- `Carding.biz_2009.11`: solo 3.076 posts / 519 usuarios.

Mezclarlos sin darnos cuenta produce varios **artefactos de análisis que parecen hallazgos y no
lo son**:

- **Comunidades de Leiden ≈ número de foros**: si detectamos ~5-6 comunidades sobre el dataset
  mezclado, es sospechosamente parecido al número de foros — probablemente estamos separando por
  foro de origen (usuarios de foros distintos casi nunca coinciden en un hilo), no por estructura
  social real dentro de una comunidad.
- **"Huecos" temporales que son en realidad fronteras entre dumps**: un hueco de actividad puede
  ser simplemente el punto donde termina un leak y empieza el siguiente, no un cierre real de un
  foro.
- **TF-IDF/LDA que detectan idioma, no tema**: si cada foro está mayoritariamente en un idioma
  distinto (alemán, cirílico, inglés genérico...), lo que "descubrimos" agrupando por foro es
  idioma, no especialización temática.

Para evitar estos artefactos — y para no hacerle spoiler a `bloque5_cardingforums`, que sí
investiga deliberadamente qué pasa al mezclar varios leaks (esa es su pregunta de investigación)
— restringimos el resto de este notebook a un único leak: **`Carder.pro_2013.04`**, el más grande,
con ventana temporal continua y el mejor porcentaje de texto válido.

In [ ]:
posts = posts[posts['forum'] == 'Carder.pro_2013.04'].copy()

print(f'Posts tras filtrar a un único foro: {len(posts):,}')
print(f'Usuarios distintos: {posts["userid"].nunique():,}')
fechas = pd.to_datetime(posts['dateline'], errors='coerce', utc=True)
print(f'Ventana temporal: {fechas.min()} -> {fechas.max()}')

## 2. Análisis de red — ¿quién habla con quién?

### ¿Qué es un grafo de co-participación?

Dos usuarios están conectados si ambos han posteado en el mismo hilo de discusión.
La idea es que participar en el mismo hilo implica algún nivel de interacción o conocimiento mutuo.

Construimos el grafo así:
1. Para cada hilo, obtenemos la lista de usuarios que postearon en él
2. Conectamos todos los pares de usuarios de esa lista entre sí
3. Repetimos para todos los hilos

El resultado es un **grafo no dirigido** (si A está conectado con B, B también lo está con A —
no hay "sentido" en la relación, porque co-participar es simétrico) donde cada nodo es un usuario
y cada arista representa co-participación en al menos un hilo.

Cada arista además tiene un **peso** (`weight`): cuántos hilos distintos comparten esos dos usuarios.
Dos usuarios que sólo coincidieron una vez, por pura casualidad en un hilo masivo, tienen una arista
débil (peso 1); dos usuarios que siempre postean juntos tienen una arista fuerte (peso alto). Ese
peso es justo lo que usa Leiden más adelante para decidir qué nodos "pertenecen" a la misma comunidad.

### Antes de programar: ¿qué hace exactamente el bucle de abajo?

Imagina 3 hilos con estos usuarios (nombres inventados para ilustrar):

```
hilo0 = [usuario0, usuario1, usuario2, usuario3, usuario4]
hilo1 = [usuario0, usuario1, usuario2, usuario3, usuario4]
hilo2 = [usuario0, usuario1, usuario2, usuario3, usuario4]
```

Dentro de un hilo, esa lista **no puede repetir usuarios** — si "usuario2" postea tres veces
en el mismo hilo, sigue contando una sola vez (por eso usamos `.unique()` antes de nada más).
El código recorre esto en dos niveles, un `for` dentro de otro `for`:

1. **Bucle exterior**: por cada hilo (`hilo0`, `hilo1`, `hilo2`, ...)
2. **Bucle interior**: dentro de ESE hilo, por cada usuario, lo compara contra los usuarios
   que vienen DESPUÉS de él en la misma lista (nunca contra los anteriores, nunca contra sí
   mismo) — así cada pareja se genera una única vez, sin duplicados.

Para `hilo0` de arriba (5 usuarios), las parejas que se generan son:

```
(usuario0, usuario1)  (usuario0, usuario2)  (usuario0, usuario3)  (usuario0, usuario4)
(usuario1, usuario2)  (usuario1, usuario3)  (usuario1, usuario4)
(usuario2, usuario3)  (usuario2, usuario4)
(usuario3, usuario4)
```

10 parejas para 5 usuarios (la fórmula es n·(n-1)/2). Si `usuario0` y `usuario1` también
coinciden en `hilo1`, no se crea una arista nueva — se suma 1 al peso de la arista que ya
existía entre ellos.

Vamos a construirlo paso a paso con datos reales para verlo en acción, antes de correrlo
sobre los 5.000 hilos completos.

#### Paso 1: quedarnos con los 5.000 hilos más grandes

Limitamos a los primeros 5.000 hilos para que el grafo sea manejable en RAM.

In [3]:
thread_col = 'topic_id' if 'topic_id' in posts.columns else 'threadid'
user_col   = 'userid'

posts = posts.dropna(subset=[thread_col, user_col])

# Tomar los primeros 5000 hilos únicos
top_threads = posts[thread_col].value_counts().head(5000).index
df = posts[posts[thread_col].isin(top_threads)]

print(f'Hilos seleccionados: {df[thread_col].nunique():,}')
print(f'Posts en esos hilos: {len(df):,}')

Hilos seleccionados: 5,000
Posts en esos hilos: 658,466


#### Paso 2: agrupar por hilo y mirar UN hilo de ejemplo

`.groupby(thread_col)` agrupa todas las filas (posts) que comparten el mismo hilo. Antes de
procesar los 5.000 hilos, miremos solo el primero para ver qué contiene exactamente.

In [4]:
grouped = df.groupby(thread_col)

# Tomamos el primer hilo como ejemplo para ver qué contiene antes de procesar los 5.000
example_thread_id, example_group = next(iter(grouped))

# .unique() quita usuarios repetidos dentro del mismo hilo; .tolist() lo convierte
# en una lista normal de Python para poder recorrerla con índices (i, j).
example_users = example_group[user_col].unique().tolist()

print(f'Hilo de ejemplo: {example_thread_id}')
print(f'Usuarios en ese hilo (sin repetidos): {example_users}')

Hilo de ejemplo: 5.0
Usuarios en ese hilo (sin repetidos): [1.0, 9129.0, 2447.0, 7269.0, 7047.0, 2621.0, 7624.0, 2669.0, 15415.0, 5850.0, 3157.0, 2798.0, 11917.0, 51014.0, 58164.0, 58189.0, 7.0]


#### Paso 3: generar las parejas de ese único hilo

Aplicamos el bucle doble (`for i` / `for j in range(i + 1, ...)`) SOLO a este hilo de ejemplo,
para ver exactamente qué parejas genera antes de repetirlo sobre los 5.000 hilos completos.

In [5]:
# 'i' recorre cada usuario del hilo, y 'j' recorre los usuarios que vienen DESPUÉS
# de 'i' en la misma lista — así generamos cada pareja una sola vez, sin repetir
# ni comparar un usuario consigo mismo.
example_pairs = []
for i in range(len(example_users)):
    for j in range(i + 1, len(example_users)):
        example_pairs.append((example_users[i], example_users[j]))

print(f'{len(example_users)} usuarios → {len(example_pairs)} parejas únicas')
for pair in example_pairs:
    print(' ', pair)

17 usuarios → 136 parejas únicas
  (1.0, 9129.0)
  (1.0, 2447.0)
  (1.0, 7269.0)
  (1.0, 7047.0)
  (1.0, 2621.0)
  (1.0, 7624.0)
  (1.0, 2669.0)
  (1.0, 15415.0)
  (1.0, 5850.0)
  (1.0, 3157.0)
  (1.0, 2798.0)
  (1.0, 11917.0)
  (1.0, 51014.0)
  (1.0, 58164.0)
  (1.0, 58189.0)
  (1.0, 7.0)
  (9129.0, 2447.0)
  (9129.0, 7269.0)
  (9129.0, 7047.0)
  (9129.0, 2621.0)
  (9129.0, 7624.0)
  (9129.0, 2669.0)
  (9129.0, 15415.0)
  (9129.0, 5850.0)
  (9129.0, 3157.0)
  (9129.0, 2798.0)
  (9129.0, 11917.0)
  (9129.0, 51014.0)
  (9129.0, 58164.0)
  (9129.0, 58189.0)
  (9129.0, 7.0)
  (2447.0, 7269.0)
  (2447.0, 7047.0)
  (2447.0, 2621.0)
  (2447.0, 7624.0)
  (2447.0, 2669.0)
  (2447.0, 15415.0)
  (2447.0, 5850.0)
  (2447.0, 3157.0)
  (2447.0, 2798.0)
  (2447.0, 11917.0)
  (2447.0, 51014.0)
  (2447.0, 58164.0)
  (2447.0, 58189.0)
  (2447.0, 7.0)
  (7269.0, 7047.0)
  (7269.0, 2621.0)
  (7269.0, 7624.0)
  (7269.0, 2669.0)
  (7269.0, 15415.0)
  (7269.0, 5850.0)
  (7269.0, 3157.0)
  (7269.0, 2798.0)
 

#### Paso 4: repetir para los 5.000 hilos y construir el grafo

Mismo bucle doble del paso anterior, pero ahora recorriendo TODOS los hilos seleccionados.
En vez de solo imprimir las parejas, las usamos para construir el grafo: cada pareja nueva
crea una arista de peso 1; si la pareja ya existía (coincidieron también en otro hilo),
sumamos 1 al peso de esa arista en vez de crear una arista duplicada.

In [6]:
# Construir el grafo con el mismo bucle doble del paso 3, aplicado a los 5.000 hilos
G = nx.Graph()

for thread_id, group in grouped:
    users_in_thread = group[user_col].unique().tolist()
    for i in range(len(users_in_thread)):
        for j in range(i + 1, len(users_in_thread)):
            u, v = str(users_in_thread[i]), str(users_in_thread[j])
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1  # más hilos compartidos = arista más fuerte
            else:
                G.add_edge(u, v, weight=1)

print(f'Nodos (usuarios): {G.number_of_nodes():,}')
print(f'Aristas (co-participaciones): {G.number_of_edges():,}')

Nodos (usuarios): 13,448
Aristas (co-participaciones): 5,991,524


#### Paso 5: visualizar el grafo recién construido

Antes de calcular ninguna métrica, echemos un vistazo a lo que acabamos de construir. Esto
**no** es la visualización final (esa llega más adelante, coloreada por comunidades) — es solo
un vistazo rápido y preliminar para confirmar que el grafo tiene la forma que esperamos: un
núcleo denso de usuarios muy conectados entre sí, y una periferia de usuarios con pocas
conexiones.

Usamos **Plotly** (igual que el resto de visualizaciones del notebook, en vez de un
`matplotlib` estático) para que las burbujas no se amontonen unas sobre otras y el resultado sea
legible. Nos quedamos con los 200 nodos de mayor grado para que el dibujo no sea un amasijo
ilegible — recordad que `G` tiene miles de nodos en total.

In [ ]:
# Vistazo preliminar: los 200 nodos con más conexiones directas (degree "en bruto",
# G.degree(), no la degree centrality normalizada que calcularemos en la siguiente celda).
preview_degree = dict(G.degree())
preview_top = sorted(preview_degree, key=preview_degree.get, reverse=True)[:200]
preview_sub = G.subgraph(preview_top)

# Mismo patrón que la visualización final de más abajo: layout con bastante separación (k alto,
# muchas iterations) para que las burbujas no se solapen.
preview_pos = nx.spring_layout(preview_sub, seed=42, k=2, iterations=100)

preview_edge_x, preview_edge_y = [], []
for u, v in preview_sub.edges():
    x0, y0 = preview_pos[u]; x1, y1 = preview_pos[v]
    preview_edge_x += [x0, x1, None]; preview_edge_y += [y0, y1, None]

preview_node_x = [preview_pos[n][0] for n in preview_sub.nodes()]
preview_node_y = [preview_pos[n][1] for n in preview_sub.nodes()]
# Aquí el degree es un conteo en bruto (puede ser miles), no un valor normalizado entre 0 y 1
# como la degree centrality — por eso el multiplicador es mucho más pequeño (0.005 en vez de 40).
preview_sizes = [preview_degree[n] * 0.005 + 5 for n in preview_sub.nodes()]

fig = go.Figure()
fig.add_trace(go.Scatter(x=preview_edge_x, y=preview_edge_y, mode='lines',
                          line=dict(width=0.3, color='#888'), hoverinfo='none'))
fig.add_trace(go.Scatter(x=preview_node_x, y=preview_node_y, mode='markers',
                          marker=dict(size=preview_sizes, color='#4C72B0'),
                          hoverinfo='skip'))
fig.update_layout(title='Vistazo preliminar del grafo — 200 usuarios más conectados',
                  showlegend=False, height=600,
                  xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
fig.show()

### Métricas de centralidad

Ahora que ya vemos el grafo que acabamos de construir (paso 5 de arriba), vamos a analizarlo con métricas de centralidad: ¿quién ocupa una posición importante dentro de esta red, y por qué?

**Degree centrality**: ¿con cuántos usuarios distintos ha coincidido este usuario? Un degree alto significa que el usuario es muy activo y alcanza a muchos otros. Es la métrica de "popularidad" más simple: solo mira conexiones directas, sin importar la forma general de la red.

**Betweenness centrality**: ¿cuántas rutas cortas entre otros pares de usuarios pasan por este nodo? Imagina que quieres ir de un usuario A a un usuario C dentro del grafo — la ruta más corta probablemente atraviese ciertos nodos intermedios. Un valor de betweenness alto indica que este usuario actúa de **puente**: si lo eliminas del grafo, grupos que antes estaban conectados quedarían separados.

La diferencia práctica importa: un usuario puede tener degree bajo (pocas conexiones directas) y aun así betweenness alto, si esas pocas conexiones son justo las que unen a dos subcomunidades que si no, no hablarían entre sí. Ese perfil — pocas conexiones pero estratégicas — es exactamente el de un intermediario o "broker": alguien que conecta partes de la red que de otro modo estarían aisladas. Estos son los actores más interesantes para inteligencia, muchas veces más que el usuario más "popular" por degree.

In [7]:
# Trabajamos sobre el componente gigante (el subgrafo más grande)
giant = G.subgraph(max(nx.connected_components(G), key=len)).copy()
print(f'Componente gigante: {giant.number_of_nodes():,} nodos ({giant.number_of_nodes()/G.number_of_nodes()*100:.1f}% del total)')

degree     = nx.degree_centrality(giant)
# Betweenness sobre muestra de 500 nodos para ir rápido
betweenness = nx.betweenness_centrality(giant, k=min(500, giant.number_of_nodes()), normalized=True)

# dict(zip(a, b)) empareja cada elemento de 'a' con el elemento en la misma posición
# de 'b', y arma un diccionario {id_usuario: nombre_usuario}. Es la forma rápida de
# pandas/Python para construir una tabla de traducción id -> nombre.
uid_to_name = dict(zip(users['userid'].astype(str), users.get('username', users.get('name', users.index.astype(str)))))

# sorted(..., key=lambda x: x[1], reverse=True) ordena la lista de pares (usuario, valor)
# de mayor a menor según el segundo elemento de cada par (el valor de betweenness).
# 'lambda x: x[1]' es una función corta y anónima que dice "dame el segundo elemento de x".
# [:10] se queda solo con los primeros 10 después de ordenar.
top_bw = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]
print('\nTop 10 brokers (betweenness):')
for uid, bw in top_bw:
    # .get(uid, uid) busca uid en el diccionario; si no lo encuentra, usa el propio
    # uid como valor por defecto en vez de fallar con un error.
    name = uid_to_name.get(uid, uid)
    print(f'  {name:<25} {bw:.4f}')

Componente gigante: 13,446 nodos (100.0% del total)



Top 10 brokers (betweenness):
  0.0                       0.0700
  ADVs                      0.0268
  Ninja                     0.0267
  makaka                    0.0093
  Devi                      0.0063
  Vasders                   0.0063
  Mr.Medved                 0.0058
  xA! | Kill yourself       0.0053
  zbox                      0.0053
  lastexile                 0.0048


### Detección de comunidades — ¿qué opciones tenemos?

En una red existen grupos de nodos que están más densamente conectados entre sí que con el
resto de la red. A estos grupos los llamamos **comunidades**.

Para detectarlas existen varios algoritmos conocidos, cada uno con su propia estrategia para
optimizar la **modularidad** (una medida de qué tan bien separadas están las comunidades entre
sí: modularidad más alta = comunidades más nítidas, más internamente conectadas que hacia
afuera). Los cuatro que vamos a comparar son:

- **Louvain**: el clásico, muy popular, rápido y con una dependencia (`python-louvain`) muy ligera.
- **Leiden**: evolución directa de Louvain, pensada para corregir un fallo conocido de este último.
- **Label Propagation**: cada nodo adopta iterativamente la etiqueta (comunidad) más común entre
  sus vecinos, hasta que el sistema se estabiliza.
- **Infomap**: basado en teoría de la información — modela paseos aleatorios (*random walks*)
  sobre el grafo y busca la partición que mejor comprime la descripción de esos paseos.

En vez de elegir uno a ciegas porque "es el que se usa en el curso", vamos a **investigarlo
empíricamente primero**: corremos los 4 algoritmos sobre el mismo grafo y comparamos cuántas
comunidades encuentra cada uno, qué modularidad consigue y cuánto tarda. De esa comparación
sacamos la conclusión — no al revés.

Usamos una muestra aleatoria más pequeña que el grafo completo (2.500 usuarios al azar, no los
más conectados) para que las 4 ejecuciones corran en segundos en clase. Ojo con esto: si
muestreamos por los usuarios con MÁS conexiones en vez de al azar, obtenemos un grafo casi
completo (todo el mundo conectado con todo el mundo) donde varios algoritmos colapsan a una
única comunidad — no es una diferencia real entre algoritmos, es un artefacto de cómo
muestreamos. Al azar, la densidad de la muestra se parece a la del grafo completo.

No hace falta entender el código de los 4 a fondo — nos vamos a quedar con Leiden para el resto
del curso, así que es al desglose de Leiden (más abajo) al que hay que prestar atención línea a
línea.

In [9]:
import time
import random
from community import community_louvain
from infomap import Infomap

# Muestra ALEATORIA (no por degree) para que la densidad se parezca a la del grafo
# completo — ver la explicación en la celda de markdown de arriba.
random.seed(42)
sample_nodes = random.sample(list(giant.nodes()), 2500)
sample_sub = giant.subgraph(sample_nodes).copy()
sample_sub = sample_sub.subgraph(max(nx.connected_components(sample_sub), key=len)).copy()
print(f'Muestra para comparar algoritmos: {sample_sub.number_of_nodes():,} nodos, {sample_sub.number_of_edges():,} aristas')

timings = {}
sample_ig = ig.Graph.from_networkx(sample_sub)

t0 = time.time()
leiden_cmp = leidenalg.find_partition(sample_ig, leidenalg.ModularityVertexPartition, weights='weight', seed=42)
timings['Leiden'] = time.time() - t0

t0 = time.time()
louvain_partition = community_louvain.best_partition(sample_sub, weight='weight', random_state=42)
timings['Louvain'] = time.time() - t0

t0 = time.time()
lp_communities = list(nx.community.label_propagation_communities(sample_sub))
timings['Label Propagation'] = time.time() - t0

t0 = time.time()
im = Infomap(silent=True)
node_to_idx = {n: i for i, n in enumerate(sample_sub.nodes())}
idx_to_node = {i: n for n, i in node_to_idx.items()}
for u, v, data in sample_sub.edges(data=True):
    im.add_link(node_to_idx[u], node_to_idx[v], data.get('weight', 1))
im.run()
timings['Infomap'] = time.time() - t0

# nx.community.modularity espera una lista de sets (una por comunidad), no un
# diccionario {nodo: id} — esta función convierte cualquier partición al mismo
# formato para que la comparación de modularidad sea justa entre las 4 librerías.
def _as_sets(membership_dict):
    groups = {}
    for node, comm in membership_dict.items():
        groups.setdefault(comm, set()).add(node)
    return list(groups.values())

leiden_names = sample_ig.vs['_nx_name']
leiden_sets = _as_sets({leiden_names[i]: leiden_cmp.membership[i] for i in range(len(leiden_names))})
infomap_sets = _as_sets({idx_to_node[node.node_id]: node.module_id for node in im.tree if node.is_leaf})

comparison = pd.DataFrame([
    ('Leiden',             len(leiden_sets),    nx.community.modularity(sample_sub, leiden_sets),    timings['Leiden']),
    ('Louvain',            len(set(louvain_partition.values())), community_louvain.modularity(louvain_partition, sample_sub), timings['Louvain']),
    ('Label Propagation',  len(lp_communities), nx.community.modularity(sample_sub, lp_communities), timings['Label Propagation']),
    ('Infomap',            len(infomap_sets),   nx.community.modularity(sample_sub, infomap_sets),   timings['Infomap']),
], columns=['Algoritmo', 'Comunidades', 'Modularidad', 'Tiempo (s)']).round(3)

comparison

Muestra para comparar algoritmos: 2,490 nodos, 208,701 aristas


,Algoritmo,Comunidades,Modularidad,Tiempo (s)
0,Leiden,6,0.385,0.634
1,Louvain,5,0.382,1.896
2,Label Propagation,5,0.000,0.257
3,Infomap,22,0.308,0.391


### ¿Por qué nos quedamos con Leiden?

Mirando la tabla de arriba: los cuatro algoritmos encuentran una estructura de comunidades
razonable sobre esta muestra, pero **Leiden** es, de los cuatro, el único que además tiene una
garantía teórica de la que carecen Louvain y Label Propagation — y es esa garantía, no solo el
número de la tabla, lo que nos hace quedarnos con él para el resto del curso.

**Leiden es una evolución directa de Louvain que corrige un fallo documentado**: Louvain puede
terminar con comunidades **internamente desconectadas** — un nodo "puente" puede quedar
asignado a una comunidad de la que, en la práctica, ya no forma parte conectada, porque el
algoritmo agrega el grafo sin comprobarlo (Traag, Waltman & van Eck, 2019, *"From Louvain to
Leiden: guaranteeing well-connected communities"*). Leiden añade una fase de refinamiento extra
antes de esa agregación que garantiza que toda comunidad detectada esté realmente bien
conectada — y además suele ser más rápido que Louvain en la práctica, como confirma la columna
"Tiempo (s)" de la tabla anterior. La única razón por la que Louvain sigue siendo tan popular es
que `python-louvain` es una dependencia más ligera; Leiden necesita `igraph` + `leidenalg`, que
también instalamos sin problema con `uv`.

A diferencia de un algoritmo como k-means, Leiden no asume que los nodos viven en un espacio de
coordenadas donde "distancia" tiene sentido — trabaja directamente sobre las conexiones (y sus
pesos) del grafo. Por eso es la elección natural cuando lo que tenemos es una red de
quién-conecta-con-quién, y no un conjunto de puntos en un espacio.

**Conclusión**: usamos Leiden para el resto del curso, no por moda, sino porque es la opción
con mejor garantía estructural entre las cuatro que acabamos de comparar.

### Antes de programar: Leiden paso a paso sobre el grafo completo

Hasta ahora solo hemos corrido Leiden sobre la muestra pequeña de la comparativa. Vamos a
aplicarlo ahora en serio, sobre el componente gigante completo (`giant`, el mismo que usamos
para las métricas de centralidad), desglosando cada paso — porque este es el análisis con el
que nos quedamos para el resto del curso, y conviene entenderlo línea a línea.

#### Paso 1: convertir el grafo de NetworkX a igraph

`leidenalg` trabaja sobre la librería `igraph`, no directamente sobre NetworkX — así que el
primer paso es convertir. `ig.Graph.from_networkx(...)` hace la conversión y, de paso, conserva
el identificador original de cada usuario en el atributo `_nx_name` de cada vértice (lo
necesitaremos en el paso 3, para poder traducir de vuelta los índices numéricos de igraph a los
ids de usuario que ya conocemos).

In [ ]:
giant_ig = ig.Graph.from_networkx(giant)
print(f'Vértices: {giant_ig.vcount():,}  |  Aristas: {giant_ig.ecount():,}')

#### Paso 2: correr `find_partition`

`leidenalg.find_partition(...)` es la función que ejecuta el algoritmo. Le pasamos el grafo de
igraph, el objetivo a optimizar (`ModularityVertexPartition`, es decir: maximizar modularidad),
la columna de pesos (`weights='weight'`, para que dos usuarios que coincidieron en 10 hilos
cuenten más que dos que coincidieron en 1) y una semilla (`seed=42`) para que el resultado sea
reproducible entre ejecuciones.

In [ ]:
leiden_result = leidenalg.find_partition(
    giant_ig, leidenalg.ModularityVertexPartition, weights='weight', seed=42
)
print(f'Comunidades detectadas: {len(leiden_result)}')

#### Paso 3: reconstruir el diccionario `{usuario: comunidad}`

El resultado de `find_partition` (`leiden_result`) habla en términos de índices numéricos de
igraph, no de los ids de usuario originales. `leiden_result.membership` es una lista donde la
posición `i` es el id de comunidad del vértice `i`; `giant_ig.vs['_nx_name']` (que guardamos en
el paso 1) es la lista de ids de usuario en ese mismo orden. Emparejando ambas listas posición a
posición reconstruimos el mismo diccionario `{usuario: id_de_comunidad}` que usaríamos con
cualquier otro algoritmo — así el resto del notebook (y la visualización de más abajo) no
necesita saber qué algoritmo generó `partition`.

In [ ]:
node_names = giant_ig.vs['_nx_name']
partition = {node_names[i]: leiden_result.membership[i] for i in range(len(node_names))}

# Counter(partition.values()) cuenta cuántos nodos cayeron en cada id de comunidad.
community_sizes = Counter(partition.values())
size_series = pd.Series(community_sizes).sort_values(ascending=False)
print('Tamaño de comunidades (top 5):')
print(size_series.head(5).to_string())

#### Paso 4: leer la modularidad

`leiden_result.modularity` nos da directamente el valor de modularidad de la partición
encontrada — no hay que calcularlo aparte. Como referencia informal: valores por encima de
~0.3 suelen indicar una estructura de comunidad significativa (la red no es aleatoria); valores
cercanos a 0 indican que la partición encontrada no es mucho mejor que agrupar al azar.

In [ ]:
modularity = leiden_result.modularity
n_communities = len(leiden_result)
print(f'Comunidades detectadas: {n_communities}')
print(f'Modularidad: {modularity:.3f}  (> 0.3 indica estructura de comunidad significativa)')

#### ¿Tienen sentido estas comunidades?

Antes de filtrar a un único foro, esta misma pregunta tenía trampa: si el grafo mezclaba varios
leaks, encontrar ~5-6 comunidades no demostraba nada sobre la estructura social del foro — podía
ser simplemente que Leiden estuviera separando "usuarios del leak A" de "usuarios del leak B",
porque casi nunca coinciden en un mismo hilo.

Ahora que el grafo se construye sobre un único foro (`Carder.pro_2013.04`), esa duda queda
despejada por construcción: ya no hay "otro foro" con el que confundir una comunidad. Las
comunidades que aparecen arriba sí son sub-grupos de usuarios reales dentro de la misma
comunidad — probablemente asociados a qué subforos frecuentan, qué tipo de hilos responden, o
con quién suelen coincidir — y no un artefacto de haber mezclado datasets sin relación entre sí.

In [ ]:
print(f'Comunidades encontradas: {n_communities}')
print(f'Foros distintos en este subset: {posts["forum"].nunique()} (ya filtrado a uno solo)')
print(
    'Si las comunidades solo reflejaran el foro de origen, con un único foro esperaríamos, como '
    'mucho, una sola comunidad grande. Encontramos varias — señal de que ahora sí es estructura '
    'social real dentro del mismo foro, no una partición espuria entre leaks distintos.'
)

### Visualización interactiva — top 150 nodos

Visualizamos los 150 nodos con mayor degree. El **color** del nodo representa su comunidad de
Leiden (cada color = una comunidad distinta detectada arriba) y el **tamaño** representa su
degree (más conexiones = nodo más grande). Pasa el ratón sobre un nodo para ver el nombre del
usuario.

Vamos a desglosar también cómo se construye este plot paso a paso — igual que hicimos con el
grafo — porque en algún momento vosotros vais a querer generar vuestros propios grafos para
vuestros informes, y para eso hace falta saber qué tocar y dónde.

#### Paso 1: elegir qué entidades mostrar

No podemos (ni queremos) dibujar los más de 13.000 usuarios del componente gigante — sería
ilegible. Nos quedamos con un número manejable de los nodos más relevantes, en este caso los
150 con mayor degree.

**Para vuestros informes**: el número 150 no tiene nada de mágico — cambiadlo por el que os
convenga (por ejemplo, `[:50]` para un informe más enfocado y legible, o `[:300]` si tenéis
pantalla/espacio para más detalle). También podríais elegir por betweenness en vez de degree, si
lo que os interesa es destacar a los "brokers" en vez de a los usuarios más activos.

In [ ]:
top150 = sorted(degree.items(), key=lambda x: x[1], reverse=True)[:150]
top150_ids = [uid for uid, _ in top150]
sub = giant.subgraph(top150_ids).copy()
print(f'Subgrafo a visualizar: {sub.number_of_nodes()} nodos, {sub.number_of_edges()} aristas')

#### Paso 2: calcular el layout (posición de cada nodo)

`nx.spring_layout` es un algoritmo de disposición basado en física: trata las aristas como
muelles que atraen a los nodos conectados, y aplica una fuerza de repulsión entre todos los
nodos para que no se amontonen. El parámetro `k` controla esa repulsión — a más `k`, más se
separan los nodos entre sí. Con el `k=1.5` original las burbujas quedaban muy pegadas y se
solapaban; subimos `k` (y las `iterations`, para darle al algoritmo más margen para converger a
una disposición estable) para separarlas visiblemente.

In [ ]:
pos = nx.spring_layout(sub, seed=42, k=3, iterations=100)

#### Paso 3: preparar coordenadas, colores y tamaños

Antes de poder dibujar necesitamos extraer, para cada nodo y cada arista, sus valores en listas
paralelas (misma posición = mismo nodo/arista), que es como Plotly espera los datos.

**Para vuestros informes:**
- **Colores**: `node_colors` mapea cada nodo a un número (aquí, su id de comunidad de Leiden) y
  `colorscale='Rainbow'` decide la paleta. Cambiad el colorscale (`'Viridis'`, `'Blues'`,
  `'Turbo'`...) o mapead por otra variable (por ejemplo, betweenness en vez de comunidad) para
  destacar otra cosa.
- **Tamaños**: `node_sizes` es una fórmula lineal `degree * multiplicador + mínimo`. Con
  `degree.get(n, 0) * 300 + 5` las burbujas salían enormes y se solapaban entre sí; bajamos el
  multiplicador a `40` para que el tamaño siga reflejando el degree relativo, pero sin que los
  nodos más conectados invadan el espacio de sus vecinos. Si vuestro grafo tiene degrees muy
  distintos a este (por ejemplo, máximo 20 en vez de un valor normalizado cerca de 1), tendréis
  que recalibrar este multiplicador a ojo — no hay una fórmula universal, se ajusta mirando el
  resultado.

In [ ]:
edge_x, edge_y = [], []
for u, v in sub.edges():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

node_x = [pos[n][0] for n in sub.nodes()]
node_y = [pos[n][1] for n in sub.nodes()]
node_names = [uid_to_name.get(n, n) for n in sub.nodes()]
node_colors = [partition.get(n, -1) for n in sub.nodes()]
# Multiplicador bajado de 300 a 40: con 300 las burbujas se veían enormes y solapadas.
node_sizes  = [degree.get(n, 0) * 40 + 5 for n in sub.nodes()]
print(f'Tamaño de burbuja — min: {min(node_sizes):.1f}, max: {max(node_sizes):.1f}')

**Ejemplo explícito: cambiar de "color = comunidad" a "color = betweenness"**

```python
# Opción A (la que usamos en la celda de abajo): color = comunidad de Leiden
node_colors = [partition.get(n, -1) for n in sub.nodes()]

# Opción B: color = betweenness (destaca a los "brokers", no a las comunidades)
node_colors = [betweenness.get(n, 0) for n in sub.nodes()]
```

Las dos líneas producen una lista del mismo tamaño (una entrada por nodo) — solo cambia qué
variable se usa. `partition` da un entero (id de comunidad: una categoría discreta); `betweenness`
da un decimal (una medida continua). Con un colorscale cualitativo como `'Rainbow'` la opción A
se ve bien porque cada color distingue una categoría; para una variable continua como
betweenness suele verse mejor un colorscale secuencial (`'Viridis'`, `'Plasma'`), donde el
degradado de color sí comunica "más o menos", no solo "distinto grupo".

In [ ]:
node_colors_por_comunidad   = [partition.get(n, -1) for n in sub.nodes()]
node_colors_por_betweenness = [betweenness.get(n, 0) for n in sub.nodes()]

print('Por comunidad (primeros 5):   ', node_colors_por_comunidad[:5])
print('Por betweenness (primeros 5): ', [round(v, 4) for v in node_colors_por_betweenness[:5]])

#### Paso 4: construir la figura

Construimos la figura en dos capas (`go.Figure` + dos `add_trace`): primero las aristas, como
líneas finas grises de fondo; luego los nodos, como marcadores encima, coloreados y
dimensionados con lo que preparamos en el paso 3.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
                          line=dict(width=0.3, color='#888'), hoverinfo='none'))
fig.add_trace(go.Scatter(x=node_x, y=node_y, mode='markers+text',
                          marker=dict(size=node_sizes, color=node_colors, colorscale='Rainbow',
                                      showscale=False),
                          text=node_names, textposition='top center',
                          hovertemplate='<b>%{text}</b><extra></extra>'))
fig.update_layout(title='Red de co-participación — top 150 usuarios (color = comunidad Leiden)',
                  showlegend=False, height=700,
                  xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
fig.show()

## 3. Análisis temporal — ¿cuándo estaba activo el foro?

Los patrones temporales revelan mucho sobre una comunidad:
- **Ciclo de vida**: cuándo creció, cuándo murió
- **Horario de actividad**: inferencia de timezone (independientemente de lo que declare el usuario)
- **Eventos externos**: picos de actividad correlacionan con eventos reales

Recordatorio importante: filtramos posts con fechas anteriores a 2000 porque algunos registros tienen timestamp=0 (corrupto), que pandas interpreta como 1970-01-01. Comprobado sobre el dataset real: el único registro con timestamp corrupto (año 1970) pertenece a `Crdshop.su_2016.11.07`, no a `Carder.pro_2013.04` — así que en el subset que ya filtramos arriba ni siquiera aparece. Mantenemos el filtro `year >= 2000` de todos modos, como salvaguarda defensiva ante futuros datos.

In [11]:
date_col = 'post_date' if 'post_date' in posts.columns else 'dateline'

df_time = posts.copy()
# pd.to_datetime convierte la columna de fecha (que puede venir como texto o número)
# en un tipo de fecha real que pandas entiende. '.dt' es el accesor que permite
# leer partes de una fecha (año, hora, etc.) de toda la columna a la vez.
df_time['date'] = pd.to_datetime(df_time[date_col], errors='coerce', utc=True)
df_time = df_time[df_time['date'].dt.year >= 2000]  # Filtrar epoch-0

# Actividad mensual
# .set_index('date') pone la columna de fecha como índice de la tabla.
# .resample('ME') agrupa las filas por mes (Month End) — es como un .groupby
# pero especializado en fechas — y .size() cuenta cuántas filas cayeron en cada mes.
monthly = df_time.set_index('date').resample('ME').size().reset_index()
monthly.columns = ['mes', 'posts']

fig = px.bar(monthly, x='mes', y='posts',
             title='Actividad mensual del foro',
             labels={'mes': 'Mes', 'posts': 'Posts'})
fig.show()

**Ya no hay hueco 2013→2015 — porque ese hueco era la frontera entre dos leaks distintos.**
Con el dataset mezclado, la actividad caía a cero entre mayo de 2013 y junio de 2015: eso era,
literalmente, el punto donde termina el dump de `Carder.pro_2013.04` (abril de 2013) y empieza el
de `Crdshop.su_2016.11.07` (que arranca justo donde el otro se corta). No era un cierre real de
un mismo foro — era el resultado de solapar dos dumps independientes en el eje temporal.

Sobre este único foro, la actividad real es un **crecimiento sostenido** desde que arranca el
dump (octubre de 2009, con muy poco volumen los primeros meses) hasta un nivel estable y alto
durante 2011-2013, con el último mes (abril de 2013) incompleto — es donde el propio volcado se
corta, no necesariamente donde el foro dejó de existir. Seguimos sin poder afirmar con estos
datos qué pasó después de abril de 2013 (¿cierre, incautación, o simplemente el dump no cubre
más?) — es una pregunta abierta, igual que antes, pero ahora aplicada al leak correcto en vez de
a un artefacto de haber mezclado dos.

In [ ]:
# Distribución horaria — inferencia de timezone
df_time['hora'] = df_time['date'].dt.hour
# .groupby('hora').size() agrupa todos los posts por hora del día y cuenta
# cuántos hay en cada hora — el mismo patrón groupby que usamos antes con hilos.
hourly = df_time.groupby('hora').size().reset_index(name='posts')

fig = px.bar(hourly, x='hora', y='posts',
             title='Distribución horaria (UTC) — pico = timezone mayoritaria',
             labels={'hora': 'Hora UTC', 'posts': 'Posts'})
fig.show()

# .idxmax() devuelve la posición (índice) de la fila con el valor máximo de 'posts';
# .loc[...] usa ese índice para leer la columna 'hora' de esa fila concreta.
peak_hour = hourly.loc[hourly['posts'].idxmax(), 'hora']
print(f'Pico de actividad: {peak_hour}h UTC')

# Interpretación (recalculada sobre el subset de un único foro, Carder.pro_2013.04 —
# el pico sigue en 17h UTC y el valle más profundo sigue entre las 2h y las 5h UTC; apenas
# cambia respecto al dataset mezclado porque Carder.pro ya era, con diferencia, el leak más
# grande del conjunto original)
valle = hourly[(hourly['hora'] >= 2) & (hourly['hora'] <= 5)]
print(f"\nValle de actividad (2h-5h UTC): {valle['posts'].sum():,} posts en ese tramo, "
      f"frente a una media horaria de {hourly['posts'].mean():,.0f} posts.")
print(
    "Ese valle (2h-5h UTC = 3h-6h de la madrugada en horario CET/Europa central) es "
    "coherente con una comunidad mayoritariamente en huso horario europeo: casi nadie "
    "escribe de madrugada. Dicho esto, la meseta ancha entre las 12h y las 21h UTC también "
    "es compatible con solapamiento de actividad de usuarios en América — con los datos que "
    "tenemos no podemos descartar esa hipótesis, solo señalar que el patrón observado es "
    "coherente con ambas explicaciones (y probablemente con una mezcla de las dos)."
)

## 4. TF-IDF — ¿de qué habla cada comunidad?

### ¿Qué es TF-IDF?

**TF-IDF** significa *Term Frequency – Inverse Document Frequency* (frecuencia de término –
frecuencia inversa de documento). Es una medida de qué tan **característico** es un término
para un documento, comparado con el resto de los documentos.

Se calcula multiplicando dos partes:

- **TF (frecuencia de término)**: qué tan seguido aparece una palabra *dentro de este documento*.
  Si "dump" aparece 200 veces en una comunidad, su TF ahí es alto.
- **IDF (frecuencia inversa de documento)**: qué tan *rara* es esa palabra en el conjunto completo
  de documentos. Palabras como "el", "de" o "the" aparecen en casi todos los documentos por igual,
  así que no distinguen nada — su IDF es bajo. Una palabra que solo aparece en uno o dos documentos
  tiene IDF alto.

**TF-IDF alto = la palabra es frecuente aquí, pero rara en los demás** → es una palabra
característica y discriminante de este documento en particular. Una palabra con TF alto pero
IDF bajo (muy común en todos los documentos) termina con TF-IDF bajo — TF-IDF filtra
automáticamente las palabras "de relleno" sin que tengamos que armar una lista manual de
stopwords específica del dominio.

**Documento = comunidad, no foro.** Al principio del notebook filtramos a un único foro
(`Carder.pro_2013.04`), así que agrupar por `forum` ya no tiene sentido (solo quedaría un
documento). En su lugar, agrupamos el texto por la **comunidad de Leiden** que detectamos en la
sección de red — así conectamos red y texto de forma natural: ¿estas comunidades que vimos por
estructura (quién habla con quién) también se diferencian por lo que hablan? Este cruce
comunidad × texto es justo lo que `bloque5_cardingforums/03_analisis_semantico.ipynb` hace de
forma mucho más rigurosa (comunidad × BERTopic) — aquí nos quedamos en una versión simplificada,
suficiente para la demo.

> **Para quien no tiene experiencia programando:** `TfidfVectorizer` de scikit-learn hace todo
> el cálculo de TF e IDF por nosotros — solo le pasamos una lista de textos y nos devuelve una
> matriz numérica donde cada fila es un documento y cada columna es una palabra del vocabulario.

### Antes de programar: ¿qué hace cada paso del TF-IDF?

Igual que hicimos con el grafo de co-participación, antes de lanzar el cálculo sobre las
comunidades reales vamos a ver cada paso con un ejemplo diminuto e inventado: dos "comunidades"
de juguete con un puñado de palabras cada una.

#### Paso 1: un "documento" por comunidad (ejemplo de juguete)

TF-IDF necesita una colección de **documentos** para comparar. Nosotros tratamos todo el texto
de una comunidad como un documento: unimos todos los posts de sus miembros en un único string
largo. Con datos reales esto se hace agrupando por id de comunidad (`.groupby('community')`, lo
veremos en el paso 4) — aquí lo escribimos a mano con dos comunidades de juguete para ver la
forma del resultado.

In [ ]:
toy_corpus = pd.Series({
    'comunidad_0':  'dump dump dump valid track2 valid',
    'comunidad_1':  'exploit exploit valid password valid',
})
toy_corpus

#### Paso 2: vectorizar el corpus de juguete

`TfidfVectorizer` convierte cada documento (cada fila de `toy_corpus`) en un vector numérico:
una columna por palabra del vocabulario, con el peso TF-IDF de esa palabra en ese documento.
Fijaos en `'valid'`: aparece en las DOS comunidades de juguete, así que su IDF es bajo y su
TF-IDF final queda por debajo del de palabras exclusivas de una sola comunidad como `'dump'` o
`'exploit'`, aunque `'valid'` no sea menos frecuente en bruto.

In [ ]:
toy_vectorizer = TfidfVectorizer()
toy_matrix = toy_vectorizer.fit_transform(toy_corpus)

toy_df = pd.DataFrame(toy_matrix.toarray(),
                       columns=toy_vectorizer.get_feature_names_out(),
                       index=toy_corpus.index)
toy_df.round(3)

#### Paso 3: extraer los términos con mayor TF-IDF de cada documento

`.toarray()[0]` convierte la fila dispersa (sparse) de un documento en un array normal;
`.argsort()` ordena los índices de menor a mayor TF-IDF, y `[-3:][::-1]` se queda con los 3 más
altos, ya invertidos de mayor a menor. Es exactamente el mismo truco que usaremos en el paso 4
sobre las comunidades reales, solo que aquí pedimos 3 términos en vez de 10 porque el
vocabulario de juguete es minúsculo.

In [ ]:
for i, doc_name in enumerate(toy_corpus.index):
    top_idx = toy_matrix[i].toarray()[0].argsort()[-3:][::-1]
    top_terms = [toy_vectorizer.get_feature_names_out()[j] for j in top_idx]
    print(f'{doc_name}: {top_terms}')

#### Paso 4: repetir sobre las comunidades reales de Leiden

Mismo procedimiento de los pasos 1 a 3 (agrupar texto por documento, vectorizar, extraer top
términos por `argsort`), aplicado ahora a las comunidades reales detectadas por Leiden, con
parámetros adicionales que no hacían falta en el ejemplo de juguete: `max_features` (límite de
vocabulario), `stop_words` (quitar palabras vacías en inglés) y `ngram_range` (también parejas
de palabras, no solo palabras sueltas). El primer paso es traducir cada post a su comunidad,
usando el mismo diccionario `partition` que construimos en la sección de red.

In [ ]:
# Traducimos cada post a la comunidad de Leiden de su autor, usando el mismo
# diccionario 'partition' que construimos en la sección de red (usuario -> comunidad).
text_col = 'text_clean' if 'text_clean' in posts.columns else None

if text_col:
    posts['community'] = posts['userid'].astype(str).map(partition)

    # dropna(): algunos usuarios pueden no estar en 'partition' (p. ej. si quedaron fuera del
    # componente gigante) — a esos posts no les podemos asignar comunidad, así que los excluimos.
    community_corpus = (
        posts.dropna(subset=['community'])
        .groupby('community')[text_col]
        .apply(lambda textos: ' '.join(textos.dropna()))
    )
    community_sizes_posts = posts.dropna(subset=['community']).groupby('community').size()

    vectorizer = TfidfVectorizer(
        max_features=2000,
        min_df=1,
        stop_words='english',   # elimina palabras vacías en inglés ('the', 'and', ...)
        ngram_range=(1, 2),     # incluye también pares de palabras ('credit card')
    )
    tfidf_matrix = vectorizer.fit_transform(community_corpus)
    feature_names = vectorizer.get_feature_names_out()

    print('Términos más característicos por comunidad de Leiden (top 10 por TF-IDF):')
    print('=' * 60)
    for i, community_id in enumerate(community_corpus.index):
        # .toarray()[0] convierte la fila dispersa (sparse) de esta comunidad en un array
        # normal; .argsort() ordena los índices de menor a mayor TF-IDF, y [-10:][::-1]
        # se queda con los 10 más altos, ya invertidos de mayor a menor.
        top_indices = tfidf_matrix[i].toarray()[0].argsort()[-10:][::-1]
        top_terms = [feature_names[j] for j in top_indices]
        n_posts = int(community_sizes_posts.get(community_id, 0))
        print(f"\n[Comunidad {int(community_id)}  —  {n_posts:,} posts]")
        print('  ' + ', '.join(top_terms))
else:
    print("Este dataset no tiene columna de texto limpio ('text_clean');")
    print('el ejemplo de TF-IDF la requiere para poder agrupar y vectorizar por comunidad.')

## 5. LDA — ¿de qué TEMAS habla cada foro?

### ¿Qué es LDA?

**LDA** (*Latent Dirichlet Allocation*, "asignación latente de Dirichlet") descubre **temas**
(topics) dentro de una colección de textos, sin que nadie tenga que etiquetar nada a mano.

La diferencia con TF-IDF importa: TF-IDF te dice qué **palabras sueltas** son características
de un documento. LDA va un paso más allá y agrupa palabras que tienden a **aparecer juntas**
en un "tema" — por ejemplo, si "dump", "track2" y "valid" aparecen juntas muy seguido, LDA las
agrupa como un tema aunque nunca le digamos qué significan esas palabras.

La intuición (sin entrar en la fórmula): LDA asume que cada documento es una **mezcla** de unos
pocos temas, y que cada tema es una **mezcla** de unas pocas palabras que tienden a coexistir.
El algoritmo no conoce los temas de antemano — los infiere iterativamente a partir de qué
palabras coinciden en los mismos documentos una y otra vez, hasta encontrar la combinación de
temas que mejor explica el corpus completo.

A diferencia de Leiden (que sí encuentra el número de grupos solo), a LDA **hay que decirle**
cuántos temas buscar (`n_components`) — es más parecido a k-means en eso: nosotros elegimos K,
el algoritmo encuentra la mejor asignación para ese K.

> **Para quien no tiene experiencia programando:** igual que con TF-IDF, `LatentDirichletAllocation`
> de scikit-learn hace todo el cálculo por nosotros — le pasamos la misma matriz de frecuencias
> de palabras que ya construimos, y nos devuelve qué palabras componen cada tema.

### Antes de programar: ¿qué hace cada paso del LDA?

Mismo enfoque que con TF-IDF: antes de aplicarlo al dataset completo, vemos cada paso con un
ejemplo de juguete — 6 documentos diminutos e inventados que mezclan claramente dos temas
(carding y hacking), para poder comprobar a ojo si LDA los separa bien.

#### Paso 1: construir la matriz de conteos (`CountVectorizer`)

A diferencia de TF-IDF, LDA trabaja sobre **conteos** de palabras, no sobre pesos TF-IDF —
por eso usamos `CountVectorizer` en vez de `TfidfVectorizer`. El resultado es una matriz
documento x palabra donde cada celda es "cuántas veces aparece esta palabra en este documento".

In [ ]:
toy_docs = [
    'dump valid track2 cvv bank',
    'valid dump cvv bin bank',
    'cvv dump bank valid track2',
    'exploit shell rootkit hacking server',
    'rootkit exploit vulnerability hacking',
    'hacking exploit server vulnerability rootkit',
]

toy_count_vectorizer = CountVectorizer()
toy_doc_term_matrix = toy_count_vectorizer.fit_transform(toy_docs)

pd.DataFrame(toy_doc_term_matrix.toarray(),
             columns=toy_count_vectorizer.get_feature_names_out())

#### Paso 2: ajustar el modelo LDA

`n_components=2` le dice al modelo que busque exactamente 2 temas (sabemos que hay 2 porque
construimos el ejemplo así — con datos reales esto es una decisión nuestra, no algo que el
dataset nos diga). `.fit(...)` es el que hace el trabajo iterativo de encontrar qué mezcla de
palabras compone cada tema.

In [ ]:
toy_lda = LatentDirichletAllocation(n_components=2, random_state=42, max_iter=50)
toy_lda.fit(toy_doc_term_matrix)
print('Modelo ajustado sobre', toy_doc_term_matrix.shape[0], 'documentos de juguete')

#### Paso 3: interpretar los temas extraídos

`toy_lda.components_` tiene una fila por tema, con el peso de cada palabra del vocabulario en
ese tema. `.argsort()[-5:][::-1]` (el mismo truco que usamos con TF-IDF) nos da las 5 palabras
de mayor peso por tema. Si todo ha ido bien, deberíamos ver un tema con vocabulario de carding
(`dump`, `cvv`, `track2`, `bank`, `valid`) y otro con vocabulario de hacking (`exploit`,
`hacking`, `rootkit`, `vulnerability`, `server`) — sin que le hayamos dicho a LDA qué significa
ninguna palabra.

In [ ]:
for topic_idx, topic in enumerate(toy_lda.components_):
    top_words = [toy_count_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-5:][::-1]]
    print(f'Tema {topic_idx}: {top_words}')

#### Paso 4: repetir sobre una muestra real de posts

Mismo procedimiento de los pasos 1 a 3 (matriz de conteos, ajuste del modelo, extracción de
temas por `argsort`), aplicado ahora a una muestra de 3.000 posts individuales del dataset real
— aquí sí elegimos `N_TOPICS = 5` a mano, y usamos posts individuales como documentos (en vez
de un documento por comunidad, como hacíamos en TF-IDF) porque LDA necesita muchos más
documentos que TF-IDF para encontrar temas con sentido.

Una diferencia más respecto al ejemplo de juguete: aquí añadimos `token_pattern` al
`CountVectorizer` para que solo cuente palabras con letras (mínimo 3 caracteres) y descarte
tokens puramente numéricos. Sin este filtro, LDA terminaba metiendo números sueltos (números
de hilo, cantidades, años...) como si fueran "palabras de un tema", cuando en realidad no
aportan ningún significado temático.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

if text_col:
    # LDA necesita MUCHOS documentos para encontrar temas con sentido — a diferencia
    # del TF-IDF de arriba (un documento por foro completo), aquí usamos una muestra
    # de posts individuales como documentos.
    sample_texts = posts[text_col].dropna().sample(n=min(3000, posts[text_col].notna().sum()), random_state=42)

    # LDA trabaja sobre conteos de palabras (no sobre TF-IDF), por eso usamos
    # CountVectorizer aquí en vez del TfidfVectorizer de la sección anterior.
    # token_pattern excluye tokens puramente numéricos (mínimo 3 letras seguidas) para que
    # los temas no se llenen de números de hilo, cantidades, años, etc.
    count_vectorizer = CountVectorizer(max_features=1000, min_df=5, stop_words='english',
                                        token_pattern=r'\b[a-zA-Z]{3,}\b')
    doc_term_matrix = count_vectorizer.fit_transform(sample_texts)
    feature_names = count_vectorizer.get_feature_names_out()

    N_TOPICS = 5  # a diferencia de Leiden, aquí SÍ tenemos que elegir cuántos temas buscar
    lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42, max_iter=10)
    lda.fit(doc_term_matrix)

    print(f'{N_TOPICS} temas detectados sobre {doc_term_matrix.shape[0]:,} posts de muestra:\n')
    for topic_idx, topic in enumerate(lda.components_):
        # Cada 'topic' es un array con el peso de cada palabra del vocabulario en ese
        # tema; .argsort()[-10:][::-1] se queda con las 10 palabras de mayor peso.
        top_words = [feature_names[i] for i in topic.argsort()[-10:][::-1]]
        print(f'Tema {topic_idx}: ' + ', '.join(top_words))
else:
    print("Este dataset no tiene columna de texto limpio ('text_clean'); el ejemplo de LDA la requiere.")